Vocabulary

In [11]:
with open("australia_dataset_final (1).txt", "r", encoding='utf-8') as file:
    dataset = file.read()

print(dataset[:1000])

datasetLength = len(dataset)

print(f"The dataset is {datasetLength} characters long") # ~10 milion characters long

vocabulary = sorted(list(set(dataset)))
vocabularyFormatted = " ".join(vocabulary)
vocabularySize = len(vocabulary)

print(f"The vocabulary is: {vocabularyFormatted}") 

"""
 ! # $ % & ' ( ) * + , - . / 0 1 2 3 4 5 6 7 8 9 : ; < > ? @ A B C D E F G H I 
 J K L M N O P Q R S T U V W X Y Z [ \ ] _ a b c d e f g h i j k l m n o p q r s t u v w x y z 
 { | ¢ £ § © ® ° ² ³ ¶ · º ¼ ½ ¾ à â ä é ‑ – — • … ′ ″ − ❑ ⦁
 """

print(f"The vocabulary size is {vocabularySize}") # 118 

<>:18: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:18: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
C:\Users\cyb26121\AppData\Local\Temp\ipykernel_1672\2685773035.py:18: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  J K L M N O P Q R S T U V W X Y Z [ \ ] _ a b c d e f g h i j k l m n o p q r s t u v w x y z


Proclamation under the Commonwealth Powers (De Facto Relationships) Act 2006I, the Governor in and over the State of Tasmania and its Dependencies in the Commonwealth of Australia, acting with the advice of the Executive Council, by this my proclamation made under section 2 of the Commonwealth Powers (De Facto Relationships) Act 2006 fix 8 October 2008 as the day on which that Act commences.29 September 2008PETER G. UNDERWOODGovernorBy His Excellency's Command,LARA GIDDINGSMinister for JusticeDisplayed and numbered in accordance with the Rules Publication Act 1953.Notified in the Gazette on 8 October 2008This proclamation is administered in the Department of Justice.
Local Government Order 2004I make the following order under section 137(1)(b) of the Local Government Act 1993 .21 September 2004J. G. COXMinister Assisting the Premier on Local Government1. Short title    This order may be cited as the Local Government Order 2004 .2. Commencement    This order takes effect on the day on w

Encoding

In [12]:
stringToInt = { ch:i for i,ch in enumerate(vocabulary)}
intToString = { i:ch for i,ch in enumerate(vocabulary)}
encode = lambda x: [stringToInt[c] for c in x] # Encodes strings into integers
decode = lambda y: ''.join([intToString[i] for i in y]) # Decodes integers into strings

# In practice we have 'subword' or tokenizer style encoding not literal character to character encoding

print(encode("My name is isaac"))
print(decode(encode("My name is isaac")))

[44, 86, 1, 75, 62, 74, 66, 1, 70, 80, 1, 70, 80, 62, 62, 64]
My name is isaac


Tensor representation of data

In [ ]:
import torch 

datasetTensor = torch.tensor(encode(dataset), dtype=torch.long) # Encode all of the text and put it in a tensor
print(datasetTensor.shape,datasetTensor.dtype)
print(datasetTensor[:100])

# Training and testing split

n = int(0.9 * len(datasetTensor)) # First 90%

training_data = datasetTensor[:n]
validation_data = dataset[n:]


torch.Size([10584978]) torch.int64
tensor([47, 79, 76, 64, 73, 62, 74, 62, 81, 70, 76, 75,  1, 82, 75, 65, 66, 79,
         1, 81, 69, 66,  1, 34, 76, 74, 74, 76, 75, 84, 66, 62, 73, 81, 69,  1,
        47, 76, 84, 66, 79, 80,  1,  8, 35, 66,  1, 37, 62, 64, 81, 76,  1, 49,
        66, 73, 62, 81, 70, 76, 75, 80, 69, 70, 77, 80,  9,  1, 32, 64, 81,  1,
        18, 16, 16, 22, 40, 12,  1, 81, 69, 66,  1, 38, 76, 83, 66, 79, 75, 76,
        79,  1, 70, 75,  1, 62, 75, 65,  1, 76])
9526480
1058498


Training

In [ ]:
batch_size = 6 # How many independent sequences are being processed at a time
chunk_size = 12 # The context size for the amount of characters that can be predicted

training_data[:chunk_size+1]

x = training_data[:chunk_size]
y = training_data[1:chunk_size+1]

# The transformer learns from all the examples within the chunk size
# In a factorial style way. So for chunk size 10 it learns from examples of length 1,2,3 etc and their
# combinations all the way up to 10. This is more computationally efficient and it allows it to
# Learn how to predict next token/character from just 1 character all the way up to chunk size

torch.manual_seed(6767) 

def get_batch(split):
    data = training_data if split == 'train' else validation_data
    index = torch.randint(len(data) - chunk_size, (chunk_size,)) # random offsets within the training set

    x = torch.stack([data[i:i+chunk_size] for i in index])
    y = torch.stack([data[i+1:i+chunk_size +1] for i in index])
    return x,y

xBatch,yBatch = get_batch('train')
print('inputs:')
print(xBatch.shape)
print(xBatch)

print('targets:')
print(yBatch.shape)
print(yBatch)

print('-------')

for b in range(batch_size): # batch dimension
    for t in range(chunk_size): # time dimension
        context = xBatch[b, :t+1] # Context is 2 dimensions consisting of the batch size and the chunk size 
        target = yBatch[b,t]
        print(f"when input is {context.tolist()} the target is: {target}")

inputs:
torch.Size([12, 12])
tensor([[  1,  77,  76,  73,  70,  81,  70,  64,  62,  73,   1,  62],
        [  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1],
        [ 75,   1,  81,  76,   1,  81,  69,  76,  80,  66,   1,  76],
        [ 64,  66,   1,  76,  67,   1,  81,  69,  66,   1,  77,  79],
        [ 76,  82,  81,  80,  81,  62,  75,  65,  70,  75,  68,   1],
        [ 66,   1,  81,  79,  62,  75,  80,  67,  66,  79,  79,  66],
        [ 76,  75,   1,  79,  66,  73,  62,  81,  70,  75,  68,   1],
        [  1,  62,  75,  65,   8,  70,  70,   9,  81,  69,  66,   1],
        [ 75,  65, 110,  80,  66,  66,   1,  80,  82,  63,  79,  66],
        [ 76,  79,  65,  80,   1,  70,  75,   1,  64,  82,  80,  81],
        [ 66,   1,  83,   1,  54,  62,  73,  72,  66,  79,   1,  58],
        [ 63,   9,   1,  81,  69,  62,  81,   1,  81,  69,  66,   1]])
targets:
torch.Size([12, 12])
tensor([[ 77,  76,  73,  70,  81,  70,  64,  62,  73,   1,  62,  65],
        [  1,   1,   1,   1,  